# Color Transfer: From Sinkhorn to Sliced OT

This notebook explores the problem of **Color Transfer** using Optimal Transport. We want to take a source image and "retarget" its colors to match the distribution (histogram) of a target image.

We will implement two methods:
1. **Sinkhorn OT**: A soft-assignment method based on entropic regularization.
2. **Sliced OT**: A state-of-the-art method that projects the problem into 1D for massive speedups (based on the work of [Bonneel et al.](https://dcoeurjo.github.io/OTColorTransfer/)).

In [ ]:
import torch
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import time
from IPython.display import display, clear_output

def load_image(path, size=128):
    img = Image.open(path).convert('RGB')
    img = img.resize((size, size))
    return torch.from_numpy(np.array(img)).float() / 255.0

# Load our synthetic assets
src = load_image('assets/source.png')
tgt = load_image('assets/target.png')

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(src); ax[0].set_title("Source (Input)")
ax[1].imshow(tgt); ax[1].set_title("Target (Palette source)")
plt.show()

## Part 1: Sinkhorn Matching (The Foundation)

The Sinkhorn algorithm solves the Kantorovich problem by adding a small amount of entropy. It produces a **coupling matrix** $P$ where $P_{ij}$ tells us how much color from pixel $i$ should be moved to color $j$.

**Complexity**: $O(N^2)$ where $N$ is the number of pixels. Because of this, we usually only run it on a small sample of pixels.

In [ ]:
def sinkhorn(a, b, C, eps=0.01, max_iter=200):
    K = torch.exp(-C / eps)
    u = torch.ones_like(a) / a.shape[0]
    v = torch.ones_like(b) / b.shape[0]
    for _ in range(max_iter):
        u = a / (torch.matmul(K, v) + 1e-12)
        v = b / (torch.matmul(K.T, u) + 1e-12)
    return u.unsqueeze(1) * K * v.unsqueeze(0)

def color_transfer_sinkhorn(src, tgt, n_samples=500):
    X_src = src.view(-1, 3)
    X_tgt = tgt.view(-1, 3)
    
    # Sample representative colors
    idx_src = torch.randperm(X_src.shape[0])[:n_samples]
    idx_tgt = torch.randperm(X_tgt.shape[0])[:n_samples]
    xs, xt = X_src[idx_src], X_tgt[idx_tgt]
    
    # Compute OT on samples
    C = torch.cdist(xs, xt, p=2)**2
    P = sinkhorn(torch.ones(n_samples)/n_samples, torch.ones(n_samples)/n_samples, C)
    
    # Map samples (Barycentric Projection)
    xs_mapped = n_samples * (P @ xt)
    
    # For simplicity in this demo, we apply the mapping back to the whole image 
    # using the nearest mapped sample for each source pixel.
    dist_to_samples = torch.cdist(X_src, xs, p=2)
    nearest_idx = dist_to_samples.argmin(dim=1)
    result = xs_mapped[nearest_idx].view(src.shape)
    return result

start = time.time()
res_sinkhorn = color_transfer_sinkhorn(src, tgt)
print(f"Sinkhorn took {time.time() - start:.2f}s")

plt.imshow(res_sinkhorn.detach())
plt.title("Sinkhorn Result (Sampled)")
plt.show()

## Part 2: Sliced Optimal Transport (The Advanced Approach)

Sliced OT is based on a brilliant mathematical insight: **The Optimal Transport problem is trivial in 1D (just sort the values)**.

By projecting our 3D RGB points onto random 1D directions, solving the OT there, and averaging the results, we can approximate the 3D OT very accurately and much faster.

**Core Algorithm**:
1. Pick a random direction $\theta$.
2. Project source and target points onto $\theta$.
3. Sort them to find the 1D mapping.
4. Shift the source points slightly towards their target match along direction $\theta$.
5. Repeat many times.

In [ ]:
def color_transfer_sliced(src, tgt, n_iter=50):
    X_src = src.view(-1, 3).clone()
    X_tgt = tgt.view(-1, 3).clone()
    
    # If different sizes, we sample to match (though SOT can handle different sizes with modification)
    N = X_src.shape[0]
    if X_tgt.shape[0] != N:
        idx = torch.randperm(X_tgt.shape[0])[:N]
        X_tgt = X_tgt[idx]

    for i in range(n_iter):
        # 1. Random direction
        theta = torch.randn(3, 1)
        theta /= torch.norm(theta)
        
        # 2. Project
        proj_src = torch.matmul(X_src, theta).squeeze()
        proj_tgt = torch.matmul(X_tgt, theta).squeeze()
        
        # 3. Sort (This is the 1D OT solution!)
        idx_src = torch.argsort(proj_src)
        idx_tgt = torch.argsort(proj_tgt)
        
        # 4. Compute the correction (Advection)
        # We move each source point by the difference in their projected positions
        # but only in the direction of theta.
        diff = proj_tgt[idx_tgt] - proj_src[idx_src]
        
        # Assign the correction back to the original source indices
        correction = diff.unsqueeze(1) * theta.T
        X_src[idx_src] += correction
        
        if i % 10 == 0:
            print(f"Iteration {i}/{n_iter}...")
            
    return X_src.view(src.shape)

start = time.time()
res_sliced = color_transfer_sliced(src, tgt)
print(f"Sliced OT took {time.time() - start:.2f}s")

plt.imshow(res_sliced.detach())
plt.title("Sliced OT Result (Global)")
plt.show()

## Visual Comparison

| Feature | Sinkhorn (Basic) | Sliced OT (Advanced) |
| :--- | :--- | :--- |
| **Conceptual Complexity** | Medium (Optimization) | Low (Sorting + Projections) |
| **Efficiency** | $O(N^2)$ (Scaling issues) | $O(K N \log N)$ (Blazing fast) |
| **Visual Quality** | Smooth, but can lose details | Sharp, preserves full distribution |
| **Scalability** | Needs sampling | Works on full images directly |

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(20, 5))
ax[0].imshow(src); ax[0].set_title("Original")
ax[1].imshow(tgt); ax[1].set_title("Target Palette")
ax[2].imshow(res_sinkhorn); ax[2].set_title("Sinkhorn")
ax[3].imshow(res_sliced); ax[3].set_title("Sliced OT")
plt.tight_layout()
plt.show()